In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Geresa_Dengue") \
    .getOrCreate()

In [20]:
ruta_dengue = "/home/jovyan/work/data/raw/datos_abiertos_vigilancia_dengue_2000_2024.csv"

df_dengue = spark.read.csv(
    ruta_dengue, 
    header=True,       
    inferSchema=True,  
    sep=";"            
)

total_filas = df_dengue.count()
total_columnas = len(df_dengue.columns)
print(f"Volumen del dataset: {total_filas} registros y {total_columnas} columnas.")

df_dengue.printSchema()

Volumen del dataset: 1029421 registros y 14 columnas.
root
 |-- departamento: string (nullable = true)
 |-- provincia: string (nullable = true)
 |-- distrito: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- enfermedad: string (nullable = true)
 |-- ano: integer (nullable = true)
 |-- semana: integer (nullable = true)
 |-- diagnostic: string (nullable = true)
 |-- diresa: integer (nullable = true)
 |-- ubigeo: integer (nullable = true)
 |-- localcod: string (nullable = true)
 |-- edad: integer (nullable = true)
 |-- tipo_edad: string (nullable = true)
 |-- sexo: string (nullable = true)



In [21]:
from pyspark.sql.functions import col, upper
df_la_libertad = df_dengue.filter(upper(col("departamento")) == "LA LIBERTAD")

total_libertad = df_la_libertad.count()
print(f"Total de registros filtrados para La Libertad: {total_libertad}")

df_la_libertad.write.csv(
    "/home/jovyan/work/data/processed/dengue_la_libertad_limpio", 
    header=True, 
    mode="overwrite"
)

Total de registros filtrados para La Libertad: 100810


In [3]:
from pyspark.sql.functions import col, to_date, lpad, concat_ws, weekofyear, avg, sum as spark_sum, round

ruta_clima = "/home/jovyan/work/data/raw/datos_climaticos_2000_2024.csv"

df_clima = spark.read.csv(
    ruta_clima, 
    header=True, 
    inferSchema=True, 
    sep=","
)

df_clima_fechas = df_clima.withColumn(
    "fecha_exacta", 
    to_date(concat_ws("-", col("YEAR"), lpad(col("DOY"), 3, "0")), "yyyy-DDD")
)

df_clima_semanas = df_clima_fechas.withColumn("semana", weekofyear(col("fecha_exacta")))

df_clima_agrupado = df_clima_semanas.groupBy("YEAR", "semana").agg(
    round(avg("T2M"), 2).alias("temp_promedio"),
    round(avg("RH2M"), 2).alias("humedad_promedio"),
    round(spark_sum("PRECTOTCORR"), 2).alias("precipitacion_total")
).orderBy("YEAR", "semana")

print("Muestra del Dataset Climático Semanal (Zona Silver):")
df_clima_agrupado.show(5)

df_clima_agrupado.write.csv(
    "/home/jovyan/work/data/processed/clima_semanal_limpio", 
    header=True, 
    mode="overwrite"
)
print("Archivo procesado y guardado")

Muestra del Dataset Climático Semanal (Zona Silver):
+----+------+-------------+----------------+-------------------+
|YEAR|semana|temp_promedio|humedad_promedio|precipitacion_total|
+----+------+-------------+----------------+-------------------+
|2000|     1|        15.78|           68.89|               0.49|
|2000|     2|         16.3|           70.75|               1.18|
|2000|     3|        17.04|            70.9|               5.14|
|2000|     4|        16.78|            75.2|               3.66|
|2000|     5|        16.62|           76.61|               7.64|
+----+------+-------------+----------------+-------------------+
only showing top 5 rows

Archivo procesado y guardado


In [4]:
df_dengue_silver = spark.read.csv("/home/jovyan/work/data/processed/dengue_la_libertad_limpio", header=True, inferSchema=True)
df_clima_silver = spark.read.csv("/home/jovyan/work/data/processed/clima_semanal_limpio", header=True, inferSchema=True)

df_clima_silver = df_clima_silver.withColumnRenamed("YEAR", "ano")

df_gold = df_dengue_silver.join(df_clima_silver, on=["ano", "semana"], how="inner")

total_cruzado = df_gold.count()
print(f"Total de registros tras el cruce (Zona Gold): {total_cruzado}")

df_gold.select("ano", "semana", "provincia", "distrito", "edad", "temp_promedio", "precipitacion_total").show(10)

Total de registros tras el cruce (Zona Gold): 100810
+----+------+---------+--------+----+-------------+-------------------+
| ano|semana|provincia|distrito|edad|temp_promedio|precipitacion_total|
+----+------+---------+--------+----+-------------+-------------------+
|2024|     9| TRUJILLO|TRUJILLO|  20|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  32|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  48|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  41|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  35|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|   6|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  25|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  53|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  54|        18.55|               7.76|
|2024|     9| TRUJILLO|TRUJILLO|  31|        18.55|               7.76|
+----+-----